## Load cleaned data

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


In [4]:
Twitter_sentiment = pd.read_csv("Twitter_sentiment.csv")  
Twitter_sentiment.head()
Twitter_sentiment['airline_sentiment'].value_counts()


airline_sentiment
negative    9080
neutral     3057
positive    2290
Name: count, dtype: int64

### Train/validation/test split

In [5]:
X = Twitter_sentiment['text'].values
y = Twitter_sentiment['airline_sentiment'].values

In [6]:
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1111,          # ~10% of original
    random_state=42,
    stratify=y_train_val
)

print(len(X_train), len(X_val), len(X_test))


11541 1443 1443


### TD-IDF vectorization

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)     
    text = re.sub(r"@\w+", " ", text)       
    text = re.sub(r"[^a-z0-9# ]", " ", text) 
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train_clean = [clean_text(t) for t in X_train]
X_val_clean   = [clean_text(t) for t in X_val]
X_test_clean  = [clean_text(t) for t in X_test]

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_val_tfidf   = tfidf.transform(X_val_clean)
X_test_tfidf  = tfidf.transform(X_test_clean)

X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape


((11541, 12291), (1443, 12291), (1443, 12291))

In [15]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # or (1,1) vs (1,2)
    min_df=2,             # try 3 or 5
    max_df=0.9,           # slightly stricter cut
    stop_words='english'
)


### Logistics Regression training

In [16]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    max_iter=500,
    multi_class='multinomial',
    n_jobs=-1
)
log_reg.fit(X_train_tfidf, y_train)


/Users/josephakumatey/airline-nlp-env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=500, multi_class='multinomial', n_jobs=-1)

In [17]:
## Class weights in Logistics Regression
log_reg = LogisticRegression(
    max_iter=500,
    multi_class='multinomial',
    n_jobs=-1,
    class_weight='balanced'  # NEW
)
log_reg.fit(X_train_tfidf, y_train)


/Users/josephakumatey/airline-nlp-env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(class_weight='balanced', max_iter=500,
                   multi_class='multinomial', n_jobs=-1)

### Evaluation (Classification report + confusion matrix)

In [18]:
# Validation set
y_val_pred = log_reg.predict(X_val_tfidf)
print("Validation accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

# Test set
y_test_pred = log_reg.predict(X_test_tfidf)
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))


Validation accuracy: 0.7422037422037422
              precision    recall  f1-score   support

    negative       0.86      0.78      0.82       908
     neutral       0.51      0.64      0.56       306
    positive       0.71      0.74      0.72       229

    accuracy                           0.74      1443
   macro avg       0.69      0.72      0.70      1443
weighted avg       0.76      0.74      0.75      1443

Test accuracy: 0.7768537768537769
              precision    recall  f1-score   support

    negative       0.88      0.82      0.85       908
     neutral       0.57      0.68      0.62       306
    positive       0.72      0.73      0.72       229

    accuracy                           0.78      1443
   macro avg       0.72      0.74      0.73      1443
weighted avg       0.79      0.78      0.78      1443



### SVM Model and comparison

In [12]:
from sklearn.svm import LinearSVC

svm_clf = LinearSVC()
svm_clf.fit(X_train_tfidf, y_train)

y_test_pred_svm = svm_clf.predict(X_test_tfidf)
print("Test accuracy (SVM):", accuracy_score(y_test, y_test_pred_svm))
print(classification_report(y_test, y_test_pred_svm))


Test accuracy (SVM): 0.7747747747747747
              precision    recall  f1-score   support

    negative       0.83      0.89      0.86       908
     neutral       0.61      0.53      0.57       306
    positive       0.73      0.63      0.68       229

    accuracy                           0.77      1443
   macro avg       0.72      0.69      0.70      1443
weighted avg       0.77      0.77      0.77      1443



### Key takeways and limitation

##### Key takeaways
1. TF–IDF + linear models provide a strong baseline on this dataset, reaching about 77–78% test accuracy and macro F1 around 0.70–0.73 on the 3‑class sentiment task.
2. Adding class_weight='balanced' to Logistic Regression meaningfully improves recall and F1 for the minority neutral and positive classes while maintaining overall accuracy, making it the best traditional baseline in this notebook.
3. Linear SVM performs similarly but tends to favor the majority negative class more, which is less desirable when distinguishing neutral/positive sentiment is important.

##### Limitations
1. TF–IDF representations ignore word order and context beyond fixed n‑grams, which makes it hard to correctly classify sarcastic, ambiguous, or subtle tweets that require deeper language understanding.
2. The models are still affected by class imbalance and struggle the most on neutral tweets, where even humans may disagree on labels, indicating a need for richer representations and possibly more nuanced labels.
3. These baselines are trained on a static snapshot of tweets and do not handle evolving language, new hashtags, or domain drift over time, which motivates moving to transformer‑based models (DistilBERT) and, in the future, continuous re‑training strategies.